# 02_02 — Preparación cartográfica SER

Este notebook prepara las capas cartográficas utilizadas para representar el ámbito del Servicio de Estacionamiento Regulado (SER) de Madrid. La unidad espacial cambia según la fuente: polígono del límite SER, polígonos de barrios SER, líneas de bandas de aparcamiento y polígonos de superficie vial.

La granularidad temporal es estática o corresponde a la versión actual descargada de cada fuente. El notebook no construye dificultad SER, agregaciones horarias, paneles, joins finales, métricas proxy ni modelos.

Las cuatro fuentes se procesan individualmente para obtener capas geoespaciales limpias, reproducibles y adecuadas para las validaciones espaciales y visualizaciones posteriores. El callejero se utilizará como fondo vial neutro para representar las bandas reguladas y distinguir, cuando la geometría lo permita, los lados de estacionamiento de una misma calle. Su utilidad para el mapa final queda condicionada a los resultados de calidad y cobertura.

Esta primera fase se limita a la configuración del entorno y a la inspección estructural de los datos raw.

## 0. Configuración inicial

La raíz del repositorio se detecta mediante la existencia de `data_catalog.csv`. Todas las rutas se gestionan de forma relativa al repositorio para evitar dependencias del entorno de ejecución.

Las operaciones espaciales se realizan en ETRS89 / UTM zona 30N, EPSG:25830. Este sistema de referencia permite expresar áreas, longitudes y tolerancias espaciales en metros.

Las fuentes de entrada se localizan bajo `data/raw/cartografia/`. En esta fase no se escriben outputs: el objetivo es comprobar que las cuatro capas pueden leerse, normalizarse e inspeccionarse antes de tomar decisiones de limpieza.

In [ ]:
from __future__ import annotations

import tempfile
import unicodedata
import zipfile
from pathlib import Path
from typing import Any

import geopandas as gpd
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 80)


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "data_catalog.csv").exists():
            return candidate
    raise FileNotFoundError(
        "No se ha encontrado data_catalog.csv al recorrer la ruta actual y sus padres. "
        f"Ruta inicial: {current}"
    )


ROOT = find_repo_root()
CATALOG_PATH = ROOT / "data_catalog.csv"
TARGET_CRS = "EPSG:25830"

TARGET_DATASET_IDS = [
    "ser_geoportal_limite_ser",
    "ser_geoportal_barrios_ser",
    "ser_geoportal_bandas_aparcamiento",
    "callejero_viales_vigentes",
]

CARTOGRAPHY_RAW_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser.geojson"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser.geojson"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/raw/cartografia/ser_geoportal_bandas_aparcamiento/"
        / "SHP_ZIP.zip"
    ),
    "callejero_viales_vigentes": (
        ROOT
        / "data/raw/cartografia/callejero_viales_vigentes/"
        / "contexto_callejero_viales_vigentes__actual.zip"
    ),
}

CANDIDATE_INTERIM_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser_clean.parquet"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser_clean.parquet"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/interim/cartografia/ser_geoportal_bandas_aparcamiento/"
        / "ser_geoportal_bandas_aparcamiento_clean.parquet"
    ),
    "callejero_viales_vigentes": (
        ROOT
        / "data/interim/cartografia/callejero_viales_vigentes/"
        / "callejero_viales_vigentes_clean.parquet"
    ),
}

LEGACY_INTERIM_PATHS = {
    "ser_geoportal_limite_ser": (
        ROOT
        / "data/interim/ser/ser_geoportal_limite_ser/"
        / "ser_geoportal_limite_ser_clean.parquet"
    ),
    "ser_geoportal_barrios_ser": (
        ROOT
        / "data/interim/ser/ser_geoportal_barrios_ser/"
        / "ser_geoportal_barrios_ser_clean.parquet"
    ),
    "ser_geoportal_bandas_aparcamiento": (
        ROOT
        / "data/interim/ser/ser_geoportal_bandas_aparcamiento/"
        / "ser_geoportal_bandas_aparcamiento_clean.parquet"
    ),
}

expected_keys = set(TARGET_DATASET_IDS)
if set(CARTOGRAPHY_RAW_PATHS) != expected_keys:
    raise ValueError(
        "Las claves de CARTOGRAPHY_RAW_PATHS no coinciden con TARGET_DATASET_IDS. "
        f"Observado: {sorted(CARTOGRAPHY_RAW_PATHS)}; esperado: {TARGET_DATASET_IDS}"
    )
if set(CANDIDATE_INTERIM_PATHS) != expected_keys:
    raise ValueError(
        "Las claves de CANDIDATE_INTERIM_PATHS no coinciden con TARGET_DATASET_IDS. "
        f"Observado: {sorted(CANDIDATE_INTERIM_PATHS)}; esperado: {TARGET_DATASET_IDS}"
    )

expected_legacy_keys = {
    "ser_geoportal_limite_ser",
    "ser_geoportal_barrios_ser",
    "ser_geoportal_bandas_aparcamiento",
}
if set(LEGACY_INTERIM_PATHS) != expected_legacy_keys:
    raise ValueError(
        "Las claves de LEGACY_INTERIM_PATHS no coinciden con las tres fuentes Geoportal. "
        f"Observado: {sorted(LEGACY_INTERIM_PATHS)}; esperado: {sorted(expected_legacy_keys)}"
    )

print(f"ROOT: {ROOT}")
print(f"GeoPandas: {gpd.__version__}")
print(f"CRS objetivo: {TARGET_CRS}")

Si no se detecta la raíz del repositorio o GeoPandas no puede cargarse, el notebook no debe continuar. El CRS objetivo permite realizar de forma coherente las operaciones métricas necesarias para la preparación cartográfica.

Las rutas se definen explícitamente para garantizar la reproducibilidad y la trazabilidad de cada fuente y de sus futuras salidas limpias.

## 1. Fuentes de entrada y salidas previstas

El notebook mantiene un identificador estable para cada una de las cuatro fuentes cartográficas y define de forma explícita sus rutas raw y sus salidas interim previstas.

La separación entre datos raw e interim evita modificar las fuentes originales y permite que cada capa limpia se genere únicamente después de superar sus comprobaciones de estructura, calidad geométrica y utilidad para el TFM.

In [ ]:
def _relative_to_root(path: Path) -> str:
    try:
        return path.relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


def _route_error(dataset_id: str, path: Path, observed: Any, expected: str) -> ValueError:
    return ValueError(
        f"Dataset: {dataset_id}; ruta: {_relative_to_root(path)}; "
        f"valor observado: {observed}; condición esperada: {expected}"
    )


route_rows = []
for dataset_id in TARGET_DATASET_IDS:
    raw_path = CARTOGRAPHY_RAW_PATHS[dataset_id]
    candidate_path = CANDIDATE_INTERIM_PATHS[dataset_id]
    legacy_path = LEGACY_INTERIM_PATHS.get(dataset_id)

    if raw_path.suffix.lower() == ".zip" and raw_path.exists():
        with zipfile.ZipFile(raw_path) as archive:
            internal_files = [name for name in archive.namelist() if not name.endswith("/")]
        n_archivos_internos = len(internal_files)
        n_shapefiles_en_zip = sum(name.lower().endswith(".shp") for name in internal_files)
    elif raw_path.suffix.lower() in {".geojson", ".json"}:
        n_archivos_internos = 1
        n_shapefiles_en_zip = pd.NA
    else:
        n_archivos_internos = pd.NA
        n_shapefiles_en_zip = pd.NA

    route_rows.append(
        {
            "dataset_id": dataset_id,
            "archivo_raw_candidato": _relative_to_root(raw_path),
            "raw_existe": raw_path.exists(),
            "n_archivos_internos": n_archivos_internos,
            "n_shapefiles_en_zip": n_shapefiles_en_zip,
            "archivo_interim_candidato": _relative_to_root(candidate_path),
            "interim_candidato_existe": candidate_path.exists(),
            "archivo_interim_legacy": pd.NA if legacy_path is None else _relative_to_root(legacy_path),
            "interim_legacy_existe": pd.NA if legacy_path is None else legacy_path.exists(),
        }
    )

route_contract = pd.DataFrame(route_rows)

for dataset_id, raw_path in CARTOGRAPHY_RAW_PATHS.items():
    if not raw_path.exists():
        raise _route_error(dataset_id, raw_path, raw_path.exists(), "el raw candidato debe existir")

raw_path_values = list(CARTOGRAPHY_RAW_PATHS.values())
if len(set(raw_path_values)) != len(raw_path_values):
    raise ValueError(
        "Las rutas raw candidatas deben ser únicas. "
        f"Valor observado: {len(set(raw_path_values))} rutas únicas; esperado: {len(raw_path_values)}"
    )

candidate_path_values = list(CANDIDATE_INTERIM_PATHS.values())
if len(set(candidate_path_values)) != len(candidate_path_values):
    raise ValueError(
        "Las rutas interim candidatas deben ser únicas. "
        f"Valor observado: {len(set(candidate_path_values))} rutas únicas; esperado: {len(candidate_path_values)}"
    )

for dataset_id, legacy_path in LEGACY_INTERIM_PATHS.items():
    if not legacy_path.exists():
        raise _route_error(dataset_id, legacy_path, legacy_path.exists(), "el interim legacy debe existir")

for dataset_id, raw_path in CARTOGRAPHY_RAW_PATHS.items():
    if raw_path.suffix.lower() == ".zip":
        observed = route_contract.loc[
            route_contract["dataset_id"].eq(dataset_id), "n_shapefiles_en_zip"
        ].iloc[0]
        if observed != 1:
            raise _route_error(dataset_id, raw_path, observed, "el ZIP debe contener exactamente un .shp")

observed_ids = route_contract["dataset_id"].tolist()
if observed_ids != TARGET_DATASET_IDS:
    raise ValueError(
        "Los dataset_id de la tabla no coinciden con TARGET_DATASET_IDS. "
        f"Observado: {observed_ids}; esperado: {TARGET_DATASET_IDS}"
    )

route_contract

Deben aparecer exactamente cuatro fuentes y todos los archivos raw deben estar disponibles. Las rutas previstas para las salidas limpias deben ser únicas y cada archivo ZIP debe contener un único shapefile inequívoco.

La columna `interim_candidato_existe` actúa como control de estado: inicialmente los outputs no existen, pero pasarán a estar disponibles conforme se complete la limpieza de cada fuente. Su existencia no debe impedir volver a ejecutar el notebook.

Si falta algún raw, una ruta está duplicada o un ZIP no contiene un shapefile inequívoco, el proceso debe detenerse antes de leer las geometrías. Si estas condiciones se cumplen, se autoriza la inspección estructural individual de las cuatro capas.

## 2. Funciones auxiliares geoespaciales

Se definen únicamente funciones generales necesarias para mostrar rutas relativas, normalizar nombres de columnas, leer GeoJSON, leer un shapefile contenido en ZIP, asegurar EPSG:25830 y resumir estructura y calidad geométrica.

Estas funciones no limpian ninguna fuente, no eliminan registros, no reparan geometrías y no deciden columnas finales. Las funciones específicas de límite, barrios, bandas y callejero se incorporarán en fases posteriores.

In [ ]:
def relpath(path: Path) -> str:
    try:
        return Path(path).relative_to(ROOT).as_posix()
    except ValueError:
        return str(path)


def strip_accents(value: str) -> str:
    normalized = unicodedata.normalize("NFKD", value)
    return "".join(char for char in normalized if not unicodedata.combining(char))


def normalize_key(value: Any) -> str:
    text = strip_accents(str(value).strip().lower())
    chars = []
    previous_was_separator = False
    for char in text:
        if char.isalnum():
            chars.append(char)
            previous_was_separator = False
        elif not previous_was_separator:
            chars.append("_")
            previous_was_separator = True
    return "".join(chars).strip("_")


def make_unique_columns(columns: list[Any]) -> list[str]:
    seen: dict[str, int] = {}
    unique_columns = []
    for column in columns:
        base = normalize_key(column)
        if not base:
            base = "column"
        count = seen.get(base, 0)
        unique = base if count == 0 else f"{base}_{count}"
        while unique in seen:
            count += 1
            unique = f"{base}_{count}"
        seen[base] = count + 1
        seen[unique] = 1
        unique_columns.append(unique)
    return unique_columns


def normalize_geo_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(f"Se esperaba un GeoDataFrame; observado: {type(gdf).__name__}")

    geometry_name = gdf.geometry.name
    if geometry_name not in gdf.columns:
        raise ValueError("El GeoDataFrame no tiene una columna geométrica activa presente en sus columnas.")

    non_geometry_columns = [column for column in gdf.columns if column != geometry_name]
    normalized_columns = make_unique_columns(["geometry", *non_geometry_columns])
    attribute_columns = normalized_columns[1:]

    attributes = pd.DataFrame(gdf.drop(columns=[geometry_name])).copy()
    attributes.columns = attribute_columns

    geometry = gpd.GeoSeries(
        gdf.geometry.copy(),
        index=gdf.index,
        crs=gdf.crs,
        name="geometry",
    )

    result = gpd.GeoDataFrame(attributes, geometry=geometry, crs=gdf.crs)
    result = result.set_geometry("geometry")
    return result


def read_geojson(path: Path, dataset_id: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            f"gpd.read_file no devolvió un GeoDataFrame: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            "no existe una geometría activa en el GeoDataFrame leído."
        )
    return gdf


def read_shp_zip(path: Path, dataset_id: str) -> gpd.GeoDataFrame:
    with zipfile.ZipFile(path) as archive:
        shp_members = [name for name in archive.namelist() if name.lower().endswith(".shp")]
        if len(shp_members) != 1:
            raise ValueError(
                f"Dataset: {dataset_id}; ruta: {relpath(path)}; valor observado: {len(shp_members)}; "
                "condición esperada: el ZIP debe contener exactamente un .shp"
            )
        shp_member = shp_members[0]
        with tempfile.TemporaryDirectory() as tmpdir:
            archive.extractall(tmpdir)
            shp_path = Path(tmpdir) / shp_member
            if not shp_path.exists():
                raise FileNotFoundError(
                    f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
                    f"no se localizó el shapefile extraído: {shp_member}"
                )
            gdf = gpd.read_file(shp_path)

    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            f"gpd.read_file no devolvió un GeoDataFrame: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(path)}; "
            "no existe una geometría activa en el GeoDataFrame leído."
        )
    return gdf


def ensure_crs_25830(gdf: gpd.GeoDataFrame, dataset_id: str) -> gpd.GeoDataFrame:
    if gdf.crs is None:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: CRS nulo; "
            "condición esperada: CRS declarado y transformable a EPSG:25830"
        )
    try:
        epsg = gdf.crs.to_epsg()
        if epsg == 25830:
            return gdf.copy()
        return gdf.to_crs(TARGET_CRS)
    except Exception as exc:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: {gdf.crs}; "
            "condición esperada: CRS transformable a EPSG:25830"
        ) from exc


def geometry_quality_summary(gdf: gpd.GeoDataFrame, dataset_id: str) -> dict[str, Any]:
    if not isinstance(gdf, gpd.GeoDataFrame):
        raise TypeError(
            f"Dataset: {dataset_id}; se esperaba un GeoDataFrame; observado: {type(gdf).__name__}"
        )
    if gdf.geometry.name not in gdf.columns:
        raise ValueError(f"Dataset: {dataset_id}; el GeoDataFrame no tiene geometría activa.")
    if gdf.crs is None:
        raise ValueError(f"Dataset: {dataset_id}; el CRS no está declarado.")

    geometry = gdf.geometry
    non_null_geometry = geometry[geometry.notna()]
    geometry_types = sorted(non_null_geometry.geom_type.dropna().unique().tolist())
    empty_count = int(non_null_geometry.is_empty.sum())
    invalid_count = int((~non_null_geometry.is_valid).sum())

    polygon_mask = non_null_geometry.geom_type.isin(["Polygon", "MultiPolygon"])
    line_mask = non_null_geometry.geom_type.isin(["LineString", "MultiLineString"])
    area_total = non_null_geometry.loc[polygon_mask].area.sum() if polygon_mask.any() else pd.NA
    length_total = non_null_geometry.loc[line_mask].length.sum() if line_mask.any() else pd.NA

    return {
        "dataset_id": dataset_id,
        "n_filas": len(gdf),
        "n_columnas": len(gdf.columns),
        "columnas_normalizadas": list(gdf.columns),
        "crs": str(gdf.crs),
        "epsg": gdf.crs.to_epsg(),
        "tipos_geometria": geometry_types,
        "geometrias_nulas": int(geometry.isna().sum()),
        "geometrias_vacias": empty_count,
        "geometrias_invalidas": invalid_count,
        "area_total_m2_aprox": area_total,
        "longitud_total_m_aprox": length_total,
    }

## 3. Inspección estructural individual de los raw

Cada fuente se inspecciona antes de tomar decisiones de limpieza. No se presuponen esquemas: las columnas disponibles se observan después de leer cada raw y no se renombra ni deriva ninguna variable específica sin comprobar antes que existe.

La normalización realizada afecta únicamente a los nombres de columnas. No se muestran registros individuales ni geometrías. La inspección revisa estructura, CRS, tipos geométricos y problemas de calidad.

In [ ]:
GEO_RAW: dict[str, gpd.GeoDataFrame] = {}
quality_rows = []

for dataset_id in TARGET_DATASET_IDS:
    raw_path = CARTOGRAPHY_RAW_PATHS[dataset_id]
    if not raw_path.exists():
        raise FileNotFoundError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: no existe; "
            "condición esperada: el raw candidato debe existir"
        )

    suffix = raw_path.suffix.lower()
    if suffix in {".geojson", ".json"}:
        raw_gdf = read_geojson(raw_path, dataset_id)
    elif suffix == ".zip":
        raw_gdf = read_shp_zip(raw_path, dataset_id)
    else:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: extensión {raw_path.suffix}; "
            "condición esperada: .geojson, .json o .zip"
        )

    observed_columns = list(raw_gdf.columns)
    if not observed_columns:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: 0 columnas; "
            "condición esperada: columnas estructurales observables"
        )

    normalized_gdf = normalize_geo_columns(raw_gdf)
    projected_gdf = ensure_crs_25830(normalized_gdf, dataset_id)

    if projected_gdf.empty:
        raise ValueError(
            f"Dataset: {dataset_id}; ruta: {relpath(raw_path)}; valor observado: GeoDataFrame vacío; "
            "condición esperada: al menos una fila para inspección estructural"
        )

    GEO_RAW[dataset_id] = projected_gdf
    quality_rows.append(geometry_quality_summary(projected_gdf, dataset_id))

inspection_table = pd.DataFrame(quality_rows)

observed_geo_ids = list(GEO_RAW)
if observed_geo_ids != TARGET_DATASET_IDS:
    raise ValueError(
        "GEO_RAW no contiene exactamente los cuatro dataset_id esperados. "
        f"Observado: {observed_geo_ids}; esperado: {TARGET_DATASET_IDS}"
    )

if len(inspection_table) != 4:
    raise ValueError(
        "La tabla de inspección debe contener exactamente cuatro filas. "
        f"Valor observado: {len(inspection_table)}; esperado: 4"
    )

for dataset_id, gdf in GEO_RAW.items():
    observed_epsg = gdf.crs.to_epsg() if gdf.crs is not None else None
    if observed_epsg != 25830:
        raise ValueError(
            f"Dataset: {dataset_id}; valor observado: EPSG {observed_epsg}; "
            "condición esperada: EPSG:25830"
        )

inspection_table

### Lectura y criterio para continuar

La inspección confirma que las cuatro fuentes se leen correctamente y pueden expresarse en EPSG:25830.

- `ser_geoportal_limite_ser` contiene un único polígono, sin geometrías nulas, vacías o inválidas. Su estructura es coherente con su función como delimitación oficial del ámbito SER.
- `ser_geoportal_barrios_ser` contiene 67 polígonos válidos. El área bruta de la capa no debe interpretarse todavía como área SER efectiva, porque antes es necesario revisar sus atributos y distinguir los polígonos realmente pertenecientes al ámbito regulado.
- `ser_geoportal_bandas_aparcamiento` contiene 87.615 geometrías lineales, sin nulos, vacíos o geometrías inválidas. La capa es estructuralmente adecuada para iniciar la revisión de colores, plazas y geometrías de bandas.
- `callejero_viales_vigentes` contiene 9.409 geometrías poligonales o multipoligonales. Se detectan 122 geometrías nulas y 37 geometrías inválidas, por lo que su incorporación al mapa queda condicionada a diagnosticar estas incidencias y comprobar que pueden repararse o excluirse sin una pérdida espacial relevante.

La decisión global es **go condicionado**. Las tres capas Geoportal presentan una estructura suficiente para continuar con su limpieza individual. El callejero también puede mantenerse, pero requiere una fase específica de diagnóstico y reparación geométrica antes de construir el fondo vial.

La siguiente fase limpiará exclusivamente `ser_geoportal_limite_ser`, siguiendo la secuencia diagnóstico → lectura → decisión → limpieza → validación posterior → output limpio. Las demás fuentes no se modificarán hasta cerrar esta primera capa.